# 02. Data Preprocessing & Feature Engineering
This notebook demonstrates the step-by-step transformations applied to our raw data to make it machine-learning ready.
Instead of writing messy, hard-to-maintain code here, we will import our production-ready functions from `src.features.build_features` to see exactly what they do under the hood.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..') # Ensure we can import from src

from src.data.load_data import load_transactions
from src.features.build_features import (
    clean_data, 
    engineer_temporal_features, 
    scale_features, 
    encode_categorical_features, 
    select_features
)

pd.set_option('display.max_columns', None)

## 1. Load Raw Data

In [2]:
print("Loading a sample of data...")
# We load the full dataset to demonstrate the pipeline
df_raw = load_transactions('../data/raw/HI-Small_Trans.csv')
display(df_raw.head())

Loading a sample of data...


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


## 2. Data Cleaning
Dropping exact duplicate rows.

In [3]:
df_clean = clean_data(df_raw)
print(f"Raw shape: {df_raw.shape} | Cleaned shape: {df_clean.shape}")

Cleaning data...
Raw shape: (5078345, 11) | Cleaned shape: (5078336, 11)


## 3. Temporal Feature Engineering
Extracting Hour, Day of Week, Month, and IsWeekend from the Timestamp.

In [4]:
df_time = engineer_temporal_features(df_clean, 'Timestamp')
display(df_time[['Timestamp', 'Hour', 'DayOfWeek', 'Month', 'IsWeekend']].head())

Engineering temporal features...


,Timestamp,Hour,DayOfWeek,Month,IsWeekend
0,2022-09-01 00:20:00,0,3,9,0
1,2022-09-01 00:20:00,0,3,9,0
2,2022-09-01 00:00:00,0,3,9,0
3,2022-09-01 00:02:00,0,3,9,0
4,2022-09-01 00:06:00,0,3,9,0


## 4. Feature Scaling
Applying Log1p to handle the massive skewness in Amount Received and Amount Paid.

In [5]:
df_scaled = scale_features(df_time, ['Amount Received', 'Amount Paid'])
display(df_scaled[['Amount Received', 'Amount Paid']].head())

Scaling numerical features...


,Amount Received,Amount Paid
0,8.215639,8.215639
1,0.009950,0.009950
2,9.594008,9.594008
3,7.940217,7.940217
4,10.510095,10.510095


## 5. Categorical Encoding
Applying One-Hot Encoding to Payment Format and Frequency Encoding to high-cardinality IDs.

In [6]:
df_encoded = encode_categorical_features(df_scaled)
display(df_encoded.head())

Encoding categorical features...


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Is Laundering,Hour,DayOfWeek,Month,IsWeekend,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Receiving Currency_Freq,Payment Currency_Freq,From Bank_Freq,To Bank_Freq
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,8.215639,US Dollar,8.215639,US Dollar,0,0,3,9,0,False,False,False,False,True,False,0.37007,0.373187,0.016074,0.008378
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.009950,US Dollar,0.009950,US Dollar,0,0,3,9,0,False,False,True,False,False,False,0.37007,0.373187,0.000012,0.005930
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,9.594008,US Dollar,9.594008,US Dollar,0,0,3,9,0,False,False,False,False,True,False,0.37007,0.373187,0.000004,0.000003
3,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,7.940217,US Dollar,7.940217,US Dollar,0,0,3,9,0,False,False,False,False,True,False,0.37007,0.373187,0.015705,0.008245
4,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,10.510095,US Dollar,10.510095,US Dollar,0,0,3,9,0,False,False,False,False,True,False,0.37007,0.373187,0.016074,0.008378


## 6. Feature Selection (Final Output)
Dropping raw text columns to leave us with a 100% numerical dataset ready for XGBoost or Random Forests.

In [7]:
df_final = select_features(df_encoded)
display(df_final.info())
display(df_final.head())

Selecting final features...
<class 'pandas.DataFrame'>
Index: 5078336 entries, 0 to 5078344
Data columns (total 17 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Amount Received              float64
 1   Amount Paid                  float64
 2   Is Laundering                int64  
 3   Hour                         int32  
 4   DayOfWeek                    int32  
 5   Month                        int32  
 6   IsWeekend                    int64  
 7   Payment Format_Bitcoin       bool   
 8   Payment Format_Cash          bool   
 9   Payment Format_Cheque        bool   
 10  Payment Format_Credit Card   bool   
 11  Payment Format_Reinvestment  bool   
 12  Payment Format_Wire          bool   
 13  Receiving Currency_Freq      float64
 14  Payment Currency_Freq        float64
 15  From Bank_Freq               float64
 16  To Bank_Freq                 float64
dtypes: bool(6), float64(6), int32(3), int64(2)
memory usage: 435.9 MB


None

,Amount Received,Amount Paid,Is Laundering,Hour,DayOfWeek,Month,IsWeekend,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Receiving Currency_Freq,Payment Currency_Freq,From Bank_Freq,To Bank_Freq
0,8.215639,8.215639,0,0,3,9,0,False,False,False,False,True,False,0.37007,0.373187,0.016074,0.008378
1,0.009950,0.009950,0,0,3,9,0,False,False,True,False,False,False,0.37007,0.373187,0.000012,0.005930
2,9.594008,9.594008,0,0,3,9,0,False,False,False,False,True,False,0.37007,0.373187,0.000004,0.000003
3,7.940217,7.940217,0,0,3,9,0,False,False,False,False,True,False,0.37007,0.373187,0.015705,0.008245
4,10.510095,10.510095,0,0,3,9,0,False,False,False,False,True,False,0.37007,0.373187,0.016074,0.008378


## Conclusion
The data is now clean, numeric, and ready for model training. The actual heavy lifting is executed by the `run_preprocessing.py` script which saves this exact output to `data/processed/processed_transactions.parquet`.